In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

def migrate_esg_data():
    # --- STEP 1: master_sites 적재 ---
    print("🚀 1단계: master_sites 적재 중...")
    sites = [
        {"site_id": "Site A", "site_name": "대전공장"},
        {"site_id": "Site B", "site_name": "울산공장"}
    ]
    # upsert를 사용하여 중복 방지
    supabase.table("master_sites").upsert(sites).execute()
    print("✅ master_sites 완료")

    # --- STEP 2: CSV 데이터 처리 및 적재 ---
    print("🚀 2단계: 실적 데이터 가공 및 적재 중...")
    
    file_map = {
        "Site A": "data/site_a_test_raw.csv",
        "Site B": "data/site_b_test_raw.csv"
    }

    for site_id, file_name in file_map.items():
        if not os.path.exists(file_name):
            print(f"⚠️ 파일을 찾을 수 없음: {file_name}")
            continue

        df = pd.read_csv(file_name)
        
        usage_records = []
        activity_records = []

        for _, row in df.iterrows():
            reporting_date = row['date'][:10] # YYYY-MM-DD 형식 유지

            # [핵심] Metric 분리 로직 적용
            # 1. 전력 (Electricity)
            usage_records.append({
                "site_id": site_id,
                "reporting_date": reporting_date,
                "metric_name": "electricity", # 분리됨
                "unit": "MWh",               # 분리됨
                "value": row['electricity_mwh'],
                "v_status": 0
            })

            # 2. LNG
            usage_records.append({
                "site_id": site_id,
                "reporting_date": reporting_date,
                "metric_name": "lng",         # 분리됨
                "unit": "Nm3",               # 분리됨
                "value": row['lng_nm3'],
                "v_status": 0
            })

            # 3. 활동 데이터 (Production)
            activity_records.append({
                "site_id": site_id,
                "reporting_date": reporting_date,
                "production_qty": row['production_ton'],
                "unit": "Ton"
            })

        # 벌크 인서트 (대량 적재 최적화)
        if usage_records:
            supabase.table("standard_usage").insert(usage_records).execute()
        if activity_records:
            supabase.table("activity_data").insert(activity_records).execute()
        
        print(f"✅ {site_id} ({file_name}) 적재 완료")

if __name__ == "__main__":
    migrate_esg_data()

🚀 1단계: master_sites 적재 중...
✅ master_sites 완료
🚀 2단계: 실적 데이터 가공 및 적재 중...
✅ Site A (data/site_a_test_raw.csv) 적재 완료
✅ Site B (data/site_b_test_raw.csv) 적재 완료
